# 08 · Hybrid Retrieval：向量检索 + BM25

向量检索擅长找“意思相近”的内容，BM25 擅长命中 SKU、货号和专有词。混合检索把两路结果都保留下来，再用 RRF（倒数排名融合）合并排名。

In [1]:
import os
import re
from pathlib import Path
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")
api_key = os.getenv("LLM_API_KEY") or os.getenv("OPENAI_API_KEY")
base_url = os.getenv("LLM_BASE_URL")
LLM_MODEL = os.getenv("LLM_MODEL", "gpt-4o-mini")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")
client = OpenAI(api_key=api_key, base_url=base_url or None) if api_key else None

def split_markdown(text, max_chars=800):
    sections = re.split(r"\n(?=#{1,3}\s)", text)
    chunks, current = [], ""
    for section in sections:
        if current and len(current) + len(section) > max_chars:
            chunks.append(current.strip())
            current = ""
        current += section + "\n\n"
    if current.strip():
        chunks.append(current.strip())
    return chunks

def load_chunks(data_dir=Path("../data")):
    items = []
    for path in sorted(data_dir.rglob("*.md")):
        if path.name == "README.md" or "images" in path.parts or "废止" in path.name:
            continue
        for text in split_markdown(path.read_text(encoding="utf-8")):
            items.append({"source": str(path.relative_to(data_dir)), "text": text})
    return items

def embed(texts):
    if not client:
        raise RuntimeError("请先配置 LLM_API_KEY 或 OPENAI_API_KEY。")
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return np.array([item.embedding for item in response.data], dtype=float)
from collections import Counter, defaultdict
import math

In [2]:
chunks = load_chunks()
chunk_vectors = embed([item["text"] for item in chunks]) if client else None
if chunk_vectors is not None:
    chunk_vectors /= np.linalg.norm(chunk_vectors, axis=1, keepdims=True)

def tokenize(text):
    return re.findall(r"[a-zA-Z0-9_#-]+|[\u4e00-\u9fff]", text.lower())

corpus = [tokenize(item["text"]) for item in chunks]
doc_frequency = Counter(token for tokens in corpus for token in set(tokens))
avg_length = sum(len(tokens) for tokens in corpus) / len(corpus)

def bm25_search(question, top_k=10, k1=1.5, b=0.75):
    scores = []
    for tokens in corpus:
        counts = Counter(tokens)
        score = 0.0
        for token in tokenize(question):
            if token not in counts:
                continue
            df = doc_frequency[token]
            idf = math.log(1 + (len(corpus) - df + 0.5) / (df + 0.5))
            tf = counts[token]
            norm = tf + k1 * (1 - b + b * len(tokens) / avg_length)
            score += idf * tf * (k1 + 1) / norm
        scores.append(score)
    indexes = np.argsort(scores)[::-1][:top_k]
    return [(index, float(scores[index])) for index in indexes]

In [3]:
def rrf(dense_results, sparse_results, top_k=4, constant=60):
    scores = defaultdict(float)
    for rank, (index, _) in enumerate(dense_results, start=1):
        scores[index] += 1 / (constant + rank)
    for rank, (index, _) in enumerate(sparse_results, start=1):
        scores[index] += 1 / (constant + rank)
    return sorted(scores.items(), key=lambda item: item[1], reverse=True)[:top_k]

question = "SKU-JK902 Cordura 500D 的规格是什么？"
if client:
    question_vector = embed([question])[0]
    question_vector /= np.linalg.norm(question_vector)
    dense_scores = chunk_vectors @ question_vector
    dense_indexes = np.argsort(dense_scores)[::-1][:10]
    dense_results = [(index, float(dense_scores[index])) for index in dense_indexes]
    sparse_results = bm25_search(question)
    final_results = rrf(dense_results, sparse_results)
    for rank, (index, score) in enumerate(final_results, start=1):
        print(f"{rank}. {score:.4f}  {chunks[index]['source']}")
else:
    print("未运行混合检索：请配置 Embedding API 后运行")

未运行混合检索：请配置 Embedding API 后运行


混合检索不是把两个分数硬加，而是先把两路结果变成排名，再用 RRF 合并。这样不要求两种算法的原始分数处在同一个尺度。